# 已训练最终V3 · Colab恢复与演示
本notebook只有恢复/验证/推理，不训练。先选参考A100 GPU；模型与数据边界详见README。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import sys,subprocess,zipfile,json
RELEASE=Path(input('输入Drive发布目录（包含最终ZIP和offline_assets）：').strip())
DEST=Path('/content/v3_pretrained_demo')
DEST.mkdir(exist_ok=False)
with zipfile.ZipFile(RELEASE/'ecc-protected-wouaf-v3-final.zip') as z:
    for name in z.namelist():
        assert (DEST/name).resolve().is_relative_to(DEST.resolve())
    z.extractall(DEST)
REPO=DEST/'ecc-protected-wouaf-v3'
ASSETS=RELEASE/'offline_assets'
def host(script,*args):
    return subprocess.run([sys.executable,str(REPO/script)]+list(map(str,args)),check=True,cwd=REPO)
def locked(script,*args):
    return host('scripts/run_locked.py',REPO/script,*args)


In [ ]:
host('scripts/verify_files.py')
host('scripts/materialize_assets.py','--asset-root',ASSETS,'--id','frozen_environment')
host('scripts/restore_environment.py')
host('scripts/run_locked.py','--check')


In [ ]:
locked('scripts/verify_pretrained.py','--output',REPO/'outputs/demo_verification.json')


In [ ]:
rows=json.loads((REPO/'demo/manifest.json').read_text())
image=next(r for r in rows if r['group']=='positive_k11' and r['condition']=='jpeg_q90_420')
locked('infer.py','--image',REPO/image['image'],'--output',REPO/'outputs/default_V3.json')
print((REPO/'outputs/default_V3.json').read_text())


## CPU级别重算完整保存结果
仍用冻结解释器；下格不前向神经网络，不训练，不重新生成图像。

In [ ]:
locked('analysis/recompute.py','--out',REPO/'outputs/independent_statistics')


V1/V2用infer.py --version v1/v2；自行训练入口独立放training/，此demo没有训练单元。大资产按asset_manifest恢复；完整步骤、版本比较和限制见README。